# LLM-RecFusion — Task 4: 多路召回融合与马太效应评估 (MovieLens-1M 真实数据)

**目标**:
1. 加载 ALS 和 LLM 两路真实召回结果 (`als_recall_ml_1m.json`, `llm_recall_ml_1m.json`)
2. 执行覆盖率 (Item Coverage)、零曝光率 (Zero-Exposure Rate) 及基尼系数计算
3. 对比两路召回在缓解马太效应上的真实差异
4. 可视化：并排柱状图 + 长尾频次分布图

**数据流**:
```
movies.dat (全局电影 ID 集合 → 覆盖率分母)
als_recall_ml_1m.json (ALS 召回结果: {user: [item_ids]})
llm_recall_ml_1m.json (LLM 召回结果: {user: [item_ids]})
    ↓
覆盖率 / 零曝光率 / 基尼系数
    ↓
并排柱状图 + 长尾分布可视化
```

In [ ]:
# ============================================================
# Cell 0: 导入 & 配置
# ============================================================
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")

# ---------- 路径配置 ----------
BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "data" / "raw" / "ml-1m"
RESULTS_DIR = BASE_DIR / "data" / "results"
FIGURES_DIR = BASE_DIR / "images"
FIGURES_DIR.mkdir(exist_ok=True, parents=True)

print(f"✅ 项目根目录: {BASE_DIR}")
print(f"✅ 原始数据: {RAW_DIR}")
print(f"✅ 召回结果: {RESULTS_DIR}")
print(f"✅ 图片保存: {FIGURES_DIR}")

---
## 1. 加载全局电影 ID 集合 & 两路召回结果

### 数据来源
- **movies.dat**: 读取 3,883 部全部电影 ID，作为覆盖率计算的分母
- **als_recall_ml_1m.json**: ALS 路召回，格式 `{user_id: [item_id1, item_id2, ...]}`
- **llm_recall_ml_1m.json**: Qwen 双塔路召回，格式同上

In [ ]:
# ============================================================
# Cell 1: 加载数据
# ============================================================
print("=" * 60)
print("【1/4】加载全局电影集合与两路召回结果...")
print("=" * 60)

# --- 1a. 加载 movies.dat 获取全局电影集合 ---
movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    dtype={"MovieID": np.int32, "Title": str, "Genres": str},
    encoding="latin-1",
)
all_movie_ids = set(movies["MovieID"].tolist())
print(f"  movies.dat 加载完成: {len(movies):,} 部电影")

# 构建 MovieID -> Title 映射 (用于可视化标签)
movie_title_map = movies.set_index("MovieID")["Title"].to_dict()
del movies

# --- 1b. 加载 ALS 召回结果 ---
als_path = RESULTS_DIR / "als_recall_ml_1m.json"
with open(als_path, "r", encoding="utf-8") as f:
    als_recall_dict = json.load(f)
# 将 key 转回 int
als_recall_dict = {int(k): [int(i) for i in v] for k, v in als_recall_dict.items()}
print(f"\n  ALS 召回: {len(als_recall_dict)} 个用户")

# --- 1c. 加载 LLM 召回结果 ---
llm_path = RESULTS_DIR / "llm_recall_ml_1m.json"
with open(llm_path, "r", encoding="utf-8") as f:
    llm_recall_dict = json.load(f)
llm_recall_dict = {int(k): [int(i) for i in v] for k, v in llm_recall_dict.items()}
print(f"  LLM 召回: {len(llm_recall_dict)} 个用户")

n_users = len(als_recall_dict)
print(f"\n  ════════════════════════════════════════")
print(f"  全局电影池 (覆盖率分母): {len(all_movie_ids):,}")
print(f"  用户数:                   {n_users:,}")
print(f"  每用户召回数:             {len(next(iter(als_recall_dict.values()))):,}")
print(f"  ════════════════════════════════════════")

---
## 2. 马太效应分析: 覆盖率 & 基尼系数

### 评估指标定义

| 指标 | 公式 | 含义 |
|------|------|------|
| **物品覆盖率 (Coverage)** | `|召回的唯一物品| / |全局物品|` | 召回结果的物品多样性 |
| **零曝光率 (Zero-Exposure)** | `1 - Coverage` | 未被召回的物品比例 |
| **基尼系数 (Gini)** | `(2·Σ(i·y_i)) / (n·Σy_i) - (n+1)/n` | 曝光均匀度 (0=均匀, 1=极端集中) |

覆盖率越高、零曝光率越低、基尼系数越小 = 马太效应缓解越好。

In [ ]:
# ============================================================
# Cell 2: 计算覆盖率 & 零曝光率 & 基尼系数
# ============================================================
print("=" * 60)
print("【2/4】计算覆盖率、零曝光率与基尼系数...")
print("=" * 60)

TOTAL_ITEMS = len(all_movie_ids)

# ---------- 工具函数 ----------
def compute_coverage(recall_dict, total_items):
    """计算物品覆盖率 (Coverage) 和零曝光率 (Zero-Exposure Rate)"""
    all_recalled = set()
    for items in recall_dict.values():
        all_recalled.update(items)
    unique_count = len(all_recalled)
    coverage = unique_count / total_items
    zero_exposure = 1.0 - coverage
    return unique_count, coverage, zero_exposure, all_recalled

def compute_gini(recall_dict):
    """计算基尼系数 (Gini Coefficient)"""
    # 统计每个物品被召回的次数
    freq = {}
    for items in recall_dict.values():
        for item in items:
            freq[item] = freq.get(item, 0) + 1
    
    y = np.sort(list(freq.values()))  # 升序排列
    n = len(y)
    if n == 0 or y.sum() == 0:
        return 0.0
    
    weighted_sum = np.sum(np.arange(1, n + 1) * y)
    gini = (2.0 * weighted_sum) / (n * y.sum()) - (n + 1.0) / n
    return float(gini)


# ---------- ALS 指标 ----------
als_unique, als_cov, als_zero, als_all_recalled = compute_coverage(als_recall_dict, TOTAL_ITEMS)
als_gini = compute_gini(als_recall_dict)

print(f"\n  {'─'*45}")
print(f"  📊 ALS Baseline 召回")
print(f"  {'─'*45}")
print(f"  唯一召回物品:    {als_unique:>5,} / {TOTAL_ITEMS:,}")
print(f"  物品覆盖率:      {als_cov*100:.2f}%")
print(f"  零曝光率:        {als_zero*100:.2f}%")
print(f"  基尼系数:        {als_gini:.4f}")

# ---------- LLM 指标 ----------
llm_unique, llm_cov, llm_zero, llm_all_recalled = compute_coverage(llm_recall_dict, TOTAL_ITEMS)
llm_gini = compute_gini(llm_recall_dict)

print(f"\n  {'─'*45}")
print(f"  📊 LLM 语义召回")
print(f"  {'─'*45}")
print(f"  唯一召回物品:    {llm_unique:>5,} / {TOTAL_ITEMS:,}")
print(f"  物品覆盖率:      {llm_cov*100:.2f}%")
print(f"  零曝光率:        {llm_zero*100:.2f}%")
print(f"  基尼系数:        {llm_gini:.4f}")

# ---------- 对比分析 ----------
print(f"\n  {'═'*45}")
print(f"  🔍 马太效应缓解对比")
print(f"  {'═'*45}")

cov_delta = (llm_cov - als_cov) * 100
gini_delta = (llm_gini - als_gini) * 100

print(f"  覆盖率提升: {cov_delta:+.2f}%")
if cov_delta > 0:
    print(f"    → LLM 语义路召回覆盖了更多长尾物品 ✓")
else:
    print(f"    → ALS 覆盖率更高 (需排查原因)")

print(f"  基尼系数变化: {gini_delta:+.2f}%")
if gini_delta < 0:
    print(f"    → LLM 曝光分布更均匀，马太效应缓解 ✓")
else:
    print(f"    → ALS 分布更均匀 (可能因 LLM 仅依赖内容语义)")

print(f"\n  ALS 未召回但 LLM 召回的物品数: {len(llm_all_recalled - als_all_recalled):,}")
print(f"  LLM 未召回但 ALS 召回的物品数: {len(als_all_recalled - llm_all_recalled):,}")

---
## 3. 可视化: 并排柱状图 + 长尾频次分布

左图: Coverage/Zero-Exposure 并排柱状图
右图: 物品召回频次长尾分布 (Log-Log)

In [ ]:
# ============================================================
# Cell 3: 马太效应可视化
# ============================================================
print("=" * 60)
print("【3/4】生成马太效应缓解可视化...")
print("=" * 60)

# --- 3a. 准备长尾分布数据 ---
def get_sorted_frequencies(recall_dict):
    freq = {}
    for items in recall_dict.values():
        for item in items:
            freq[item] = freq.get(item, 0) + 1
    return np.sort(list(freq.values()))[::-1]  # 降序

als_freqs = get_sorted_frequencies(als_recall_dict)
llm_freqs = get_sorted_frequencies(llm_recall_dict)

# --- 3b. 创建画布 ---
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(18, 7), dpi=150)
fig.patch.set_facecolor("#0F0F1A")

DARK_BG = "#1A1A2E"
TEXT_COLOR = "#E0E0E0"
ACCENT_LINE = "#2A2A4A"

# ---------- 左图: 覆盖率并排柱状图 ----------
ax_left.set_facecolor(DARK_BG)

categories = ["Coverage", "Zero-Exposure"]
x = np.arange(len(categories))
width = 0.35

als_vals = [als_cov * 100, als_zero * 100]
llm_vals = [llm_cov * 100, llm_zero * 100]

bars1 = ax_left.bar(
    x - width / 2, als_vals, width,
    label="ALS Baseline",
    color="#4A90D9", edgecolor="#6AB0FF",
    linewidth=1.2, alpha=0.92,
)
bars2 = ax_left.bar(
    x + width / 2, llm_vals, width,
    label="LLM Dual-Tower",
    color="#00E5A0", edgecolor="#30FFC0",
    linewidth=1.2, alpha=0.92,
)

# 数值标注
for bar, val in zip(bars1, als_vals):
    ax_left.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f"{val:.2f}%",
        ha="center", va="bottom",
        fontsize=13, fontweight="bold", color=TEXT_COLOR,
    )
for bar, val in zip(bars2, llm_vals):
    ax_left.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f"{val:.2f}%",
        ha="center", va="bottom",
        fontsize=13, fontweight="bold", color=TEXT_COLOR,
    )

ax_left.set_xticks(x)
ax_left.set_xticklabels(categories, fontsize=13, color=TEXT_COLOR)
ax_left.set_ylabel("Percentage (%)", fontsize=14, color=TEXT_COLOR, labelpad=10)
ax_left.set_title(
    "Matthew Effect Mitigation:\nItem Coverage Comparison",
    fontsize=15, fontweight="bold", color=TEXT_COLOR, pad=15,
)
ax_left.legend(
    fontsize=12, facecolor=DARK_BG,
    edgecolor=ACCENT_LINE, labelcolor=TEXT_COLOR,
    loc="upper right",
)
ax_left.set_ylim(0, max(max(als_vals), max(llm_vals)) * 1.25)
ax_left.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax_left.tick_params(axis="y", colors=TEXT_COLOR, labelsize=12)
ax_left.grid(axis="y", linestyle="--", alpha=0.25, color=ACCENT_LINE)
ax_left.set_axisbelow(True)
for spine in ax_left.spines.values():
    spine.set_color(ACCENT_LINE)
    spine.set_linewidth(0.8)

# ---------- 右图: 长尾频次分布 (Log-Log) ----------
ax_right.set_facecolor(DARK_BG)

rank_als = np.arange(1, len(als_freqs) + 1)
rank_llm = np.arange(1, len(llm_freqs) + 1)

ax_right.plot(
    rank_als, als_freqs,
    color="#4A90D9", linewidth=2.0, alpha=0.85,
    label=f"ALS Baseline (Gini={als_gini:.3f})",
)
ax_right.plot(
    rank_llm, llm_freqs,
    color="#00E5A0", linewidth=2.0, alpha=0.85,
    label=f"LLM Dual-Tower (Gini={llm_gini:.3f})",
)

ax_right.set_xscale("log")
ax_right.set_yscale("log")

ax_right.set_xlabel("Item Rank (log)", fontsize=14, color=TEXT_COLOR, labelpad=10)
ax_right.set_ylabel("Recall Frequency (log)", fontsize=14, color=TEXT_COLOR, labelpad=10)
ax_right.set_title(
    "Long-tail Recall Frequency Distribution",
    fontsize=15, fontweight="bold", color=TEXT_COLOR, pad=15,
)
ax_right.legend(
    fontsize=12, facecolor=DARK_BG,
    edgecolor=ACCENT_LINE, labelcolor=TEXT_COLOR,
    loc="upper right",
)
ax_right.tick_params(axis="both", colors=TEXT_COLOR, labelsize=12)
ax_right.grid(True, which="major", linestyle="--", alpha=0.25, color=ACCENT_LINE)
ax_right.grid(True, which="minor", linestyle=":", alpha=0.10, color=ACCENT_LINE)
ax_right.set_axisbelow(True)
for spine in ax_right.spines.values():
    spine.set_color(ACCENT_LINE)
    spine.set_linewidth(0.8)

# ---------- 顶部标注 ----------
fig.text(
    0.5, 0.96,
    f"Gini Δ = {gini_delta:+.2f}%  |  "
    f"ALS Coverage: {als_cov*100:.2f}%  |  "
    f"LLM Coverage: {llm_cov*100:.2f}%  |  "
    f"ALS #Items: {als_unique:,}  |  "
    f"LLM #Items: {llm_unique:,}",
    ha="center", va="top",
    fontsize=13, fontweight="bold",
    color="#FFE066",
    transform=fig.transFigure,
)

fig.text(
    0.5, 0.92,
    "LLM-RecFusion — Matthew Effect Analysis (MovieLens-1M) | "
    "Gini ↓ means more uniform distribution (less Matthew Effect)",
    ha="center", va="top",
    fontsize=11, fontstyle="italic",
    color="#8888AA",
    transform=fig.transFigure,
)

plt.tight_layout(rect=[0, 0, 1, 0.90])

# --- 保存 ---
matthew_save_path = FIGURES_DIR / "matthew_effect_coverage.png"
fig.savefig(matthew_save_path, dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
print(f"  ✅ 图表已保存: {matthew_save_path}")

plt.show()

---
## 4. 深度分析: LLM 长尾挖掘能力

分析 LLM 语义路召回中那些 ALS **未召回**的物品，理解 LLM 在语义匹配上的独特价值。

In [ ]:
# ============================================================
# Cell 4: LLM 长尾挖掘深度分析
# ============================================================
print("=" * 60)
print("【4/4】LLM 长尾挖掘深度分析...")
print("=" * 60)

# --- 4a. 找出 LLM 独有的召回物品 ---
als_items = set()
for items in als_recall_dict.values():
    als_items.update(items)

llm_items = set()
for items in llm_recall_dict.values():
    llm_items.update(items)

llm_exclusive = llm_items - als_items  # LLM 召回但 ALS 没召回的物品

print(f"\n  LLM 独有召回 (ALS 未召回) 的物品数: {len(llm_exclusive):,}")

# --- 4b. 展示部分 LLM 独有物品 ---
print(f"\n  📝 LLM 独有召回样本 (前 15 部电影):")
print(f"  {'─'*60}")
for i, mid in enumerate(sorted(llm_exclusive)[:15]):
    title = movie_title_map.get(mid, f"Unknown ({mid})")
    print(f"  {i+1:>2}. MovieID={mid:<6} | {title[:50]}")

# --- 4c. 统计摘要 ---
print(f"\n  {'─'*50}")
print(f"  📊 马太效应缓解摘要")
print(f"  {'─'*50}")
print(f"  全局电影池:         {TOTAL_ITEMS:,}")
print(f"  ALS 召回唯一电影:   {len(als_items):,} ({len(als_items)/TOTAL_ITEMS*100:.1f}%)")
print(f"  LLM 召回唯一电影:   {len(llm_items):,} ({len(llm_items)/TOTAL_ITEMS*100:.1f}%)")
print(f"  两路重合电影:       {len(als_items & llm_items):,}")
print(f"  LLM 独有发现:       {len(llm_exclusive):,}")
print(f"  ALS 独有发现:       {len(als_items - llm_items):,}")
print(f"")
print(f"  ALS Gini:           {als_gini:.4f}")
print(f"  LLM Gini:           {llm_gini:.4f}")
print(f"  Gini Δ:             {gini_delta:+.2f}%")

print(f"\n{'=' * 60}")
print(f"✅ 多路召回融合与马太效应评估完成!")
print(f"{'=' * 60}")
print(f"\n  使用的真实数据:")
print(f"    - movies.dat (3,883 部电影, 覆盖率分母)")
print(f"    - als_recall_ml_1m.json (ALS 召回结果)")
print(f"    - llm_recall_ml_1m.json (LLM 语义召回结果)")
print(f"\n  关键发现可支撑: LLM 语义路 vs ALS 协同过滤在缓解马太效应上的能力评估")